# Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/notebooks/intro/04_analysis.ipynb)

Official API intro to `minilink.analysis`: linearization, modal / frequency tools,
structural properties, equilibria, and discretization.

**Scripts for depth:** `examples/scripts/analysis/`


In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")


## Linearize about an operating point

`linearize` returns an `LTISystem` with $A, B, C, D$ at $x_\bar$.


In [ ]:
import numpy as np
from minilink.analysis.linearize import linearize, linearize_matrices
from minilink.dynamics.catalog.pendulum.pendulum import InvertedPendulum

plant = InvertedPendulum()
x_bar = np.array([0.0, 0.0])  # upright for this model
lti = linearize(plant, x_bar)
print("A =\n", np.round(lti.A(), 4))
print("B =\n", np.round(lti.B(), 4))
print("poles:", np.round(np.linalg.eigvals(lti.A()), 2))

A, B, C, D = linearize_matrices(plant, x_bar, method="fd")
print("FD A close to AD:", np.allclose(A, lti.A(), atol=1e-4))


## Structural properties and modal analysis

Controllability / observability and modal tools operate on linearized models.


In [ ]:
from minilink.analysis.structural import controllability, observability

A, B, C = lti.A(), lti.B(), lti.C()
ctrl = controllability(A, B)
obs = observability(A, C)
print("controllable:", ctrl.is_full_rank, f"(rank {ctrl.rank}/{ctrl.n})")
print("observable:  ", obs.is_full_rank, f"(rank {obs.rank}/{obs.n})")

# Modal analysis façade on the nonlinear plant about the same operating point
plant.modal_analysis(x_bar=x_bar, mode="all")


## Frequency response

`bode` returns magnitude, phase, and frequency samples for a selected channel.


In [ ]:
from minilink.analysis.frequency import bode

mag, phase, omega = bode(plant, x_bar=x_bar)
print("omega samples:", len(omega), "mag shape:", np.shape(mag))
print("Plot demo: examples/scripts/analysis/demo_bode.py")


## Equilibria and discretization

Find steady states and map continuous LTI models to discrete-time.


In [ ]:
from minilink.analysis.equilibria import find_equilibrium
from minilink.analysis.discretize import discretize

print("find_equilibrium:", find_equilibrium)
print("discretize:", discretize)
print("Demos: examples/scripts/analysis/demo_equilibrium.py")
